# Rerun top-3 configurations
This notebook runs the top-3 configurations with multiple seeds (converted from `rerun_top3.py`).
Run cells sequentially using the kernel that has `torch` installed. For a fast sanity check set `seeds=[1]` and reduce `epochs` in Cell 5.

In [1]:
# Configuration: top-3 configs and seeds
top3 = [
    {"units": (128, 64, 32), "dropout": 0.0, "norm": "batch"},
    {"units": (256, 128, 64), "dropout": 0.0, "norm": "batch"},
    {"units": (256, 128, 64), "dropout": 0.2, "norm": "batch"},
]
seeds = [1, 2, 3]
# Quick option: set seeds=[1] and epochs=1 for a fast sanity check

In [2]:
# Imports and data setup
import torch
import torch.optim as optim
from torch import nn
from mads_datasets import DatasetFactoryProvider, DatasetType
from mltrainer import Trainer, TrainerSettings, ReportTypes, metrics
from mltrainer.preprocessors import BasePreprocessor

# Create streamers (adjust batchsize/preprocessor as needed)
fashionfactory = DatasetFactoryProvider.create_factory(DatasetType.FASHION)
preprocessor = BasePreprocessor()
streamers = fashionfactory.create_datastreamer(batchsize=64, preprocessor=preprocessor)
train = streamers['train']
valid = streamers['valid']
trainstreamer = train.stream()
validstreamer = valid.stream()

accuracy = metrics.Accuracy()
loss_fn = torch.nn.CrossEntropyLoss()

2026-06-13 18:00:32.934 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /home/fblaak/.cache/mads_datasets/fashionmnist
2026-06-13 18:00:32.935 | INFO     | mads_datasets.base:download_data:124 - File already exists at /home/fblaak/.cache/mads_datasets/fashionmnist/fashionmnist.pt


In [3]:
# Model definition (same as in the notebook)
class NeuralNetwork(nn.Module):
    def __init__(self, num_classes: int, units1: int, units2: int, units3: int, dropout_rate: float = 0.0, normalization: str = "none") -> None:
        super().__init__()
        self.num_classes = num_classes
        self.units1 = units1
        self.units2 = units2
        self.units3 = units3
        self.dropout_rate = float(dropout_rate)
        self.normalization = str(normalization)

        def block(in_features, out_features):
            layers = [nn.Linear(in_features, out_features)]
            if self.normalization == "batch":
                layers.append(nn.BatchNorm1d(out_features))
            elif self.normalization == "layer":
                layers.append(nn.LayerNorm(out_features))
            layers.append(nn.ReLU())
            if self.dropout_rate and self.dropout_rate > 0.0:
                layers.append(nn.Dropout(self.dropout_rate))
            return layers

        layers = []
        layers.extend(block(28 * 28, self.units1))
        layers.extend(block(self.units1, self.units2))
        layers.extend(block(self.units2, self.units3))
        layers.append(nn.Linear(self.units3, self.num_classes))
        self.model = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(x.size(0), -1)
        logits = self.model(x)
        return logits

In [4]:
# Runner: iterate configs and seeds.
# Warning: this will run full trainings — for a quick check set seeds=[1] and epochs=1 below.
from datetime import datetime

for cfg in top3:
    u1, u2, u3 = cfg['units']
    dr = float(cfg['dropout'])
    norm = cfg['norm']
    for seed in seeds:
        try:
            torch.manual_seed(seed)
            model = NeuralNetwork(num_classes=10, units1=u1, units2=u2, units3=u3, dropout_rate=dr, normalization=norm)
            model.experiment = f"rerun_top3_{u1}_{u2}_{u3}_dr{dr}_norm{norm}_seed{seed}"
            model.seed = int(seed)
            settings = TrainerSettings(
                epochs=30,
                metrics=[accuracy],
                logdir="modellogs",
                train_steps=100,
                valid_steps=100,
                reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
            )
            trainer = Trainer(
                model=model,
                settings=settings,
                loss_fn=loss_fn,
                optimizer=optim.Adam,
                traindataloader=trainstreamer,
                validdataloader=validstreamer,
                scheduler=optim.lr_scheduler.ReduceLROnPlateau,
            )
            print(datetime.now(), "Starting run:", model.experiment)
            trainer.loop()
            print(datetime.now(), "Finished run:", model.experiment)
        except Exception as e:
            print("Run failed:", e)
            continue

print("All reruns completed")

2026-06-13 18:00:33.004 | INFO     | mltrainer.settings:check_path:60 - Created logdir /home/fblaak/MADS-ML-PORTFOLIO-FBLAAK/portfolio-example-main/2-hypertuning-mlflow/modellogs
2026-06-13 18:00:33.004 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs/20260613-180033
2026-06-13 18:00:33.941 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


2026-06-13 18:00:33.992536 Starting run: rerun_top3_128_64_32_dr0.0_normbatch_seed1


100%|██████████| 100/100 [00:00<00:00, 215.30it/s]
2026-06-13 18:00:34.685 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.1101 test 0.7323 metric ['0.7939']
100%|██████████| 100/100 [00:00<00:00, 206.78it/s]
2026-06-13 18:00:35.395 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.6508 test 0.5673 metric ['0.8166']
100%|██████████| 100/100 [00:00<00:00, 229.12it/s]
2026-06-13 18:00:36.054 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 0.4838 test 0.4922 metric ['0.8387']
100%|██████████| 100/100 [00:00<00:00, 229.19it/s]
2026-06-13 18:00:36.713 | INFO     | mltrainer.trainer:report:209 - Epoch 3 train 0.4718 test 0.4592 metric ['0.8336']
100%|██████████| 100/100 [00:00<00:00, 228.32it/s]
2026-06-13 18:00:37.378 | INFO     | mltrainer.trainer:report:209 - Epoch 4 train 0.4622 test 0.4901 metric ['0.8302']
2026-06-13 18:00:37.379 | INFO     | mltrainer.trainer:__call__:252 - best loss: 0.4592, current loss 0.4901.Counter 1/10.
100%|██████████| 100/100 [0

2026-06-13 18:00:53.918978 Finished run: rerun_top3_128_64_32_dr0.0_normbatch_seed1
2026-06-13 18:00:53.928178 Starting run: rerun_top3_128_64_32_dr0.0_normbatch_seed2


100%|██████████| 100/100 [00:00<00:00, 223.75it/s]
2026-06-13 18:00:54.599 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.1419 test 0.8318 metric ['0.7797']
100%|██████████| 100/100 [00:00<00:00, 230.13it/s]
2026-06-13 18:00:55.256 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.6389 test 0.5688 metric ['0.8247']
100%|██████████| 100/100 [00:00<00:00, 229.06it/s]
2026-06-13 18:00:55.916 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 0.5365 test 0.5329 metric ['0.8164']
100%|██████████| 100/100 [00:00<00:00, 231.65it/s]
2026-06-13 18:00:56.569 | INFO     | mltrainer.trainer:report:209 - Epoch 3 train 0.4712 test 0.4367 metric ['0.8550']
100%|██████████| 100/100 [00:00<00:00, 231.44it/s]
2026-06-13 18:00:57.225 | INFO     | mltrainer.trainer:report:209 - Epoch 4 train 0.4499 test 0.4406 metric ['0.8431']
2026-06-13 18:00:57.228 | INFO     | mltrainer.trainer:__call__:252 - best loss: 0.4367, current loss 0.4406.Counter 1/10.
100%|██████████| 100/100 [0

...

2026-06-13 18:03:10.793393 Finished run: rerun_top3_256_128_64_dr0.2_normbatch_seed1
2026-06-13 18:03:10.803309 Starting run: rerun_top3_256_128_64_dr0.2_normbatch_seed2


100%|██████████| 100/100 [00:00<00:00, 170.76it/s]
2026-06-13 18:03:11.633 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.0433 test 0.6506 metric ['0.7945']
100%|██████████| 100/100 [00:00<00:00, 171.80it/s]
2026-06-13 18:03:12.466 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.6202 test 0.5162 metric ['0.8280']
100%|██████████| 100/100 [00:00<00:00, 171.18it/s]
2026-06-13 18:03:13.301 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 0.5458 test 0.5370 metric ['0.8158']
2026-06-13 18:03:13.302 | INFO     | mltrainer.trainer:__call__:252 - best loss: 0.5162, current loss 0.5370.Counter 1/10.
100%|██████████| 100/100 [00:00<00:00, 173.66it/s]
2026-06-13 18:03:14.132 | INFO     | mltrainer.trainer:report:209 - Epoch 3 train 0.5053 test 0.4466 metric ['0.8372']
100%|██████████| 100/100 [00:00<00:00, 172.75it/s]
2026-06-13 18:03:14.960 | INFO     | mltrainer.trainer:report:209 - Epoch 4 train 0.4851 test 0.4443 metric ['0.8414']
100%|██████████| 100/100 [0

2026-06-13 18:03:35.908121 Finished run: rerun_top3_256_128_64_dr0.2_normbatch_seed2
2026-06-13 18:03:35.916819 Starting run: rerun_top3_256_128_64_dr0.2_normbatch_seed3


100%|██████████| 100/100 [00:00<00:00, 172.12it/s]
2026-06-13 18:03:36.744 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.0217 test 0.6397 metric ['0.7913']
100%|██████████| 100/100 [00:00<00:00, 175.33it/s]
2026-06-13 18:03:37.557 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.6316 test 0.5329 metric ['0.8111']
100%|██████████| 100/100 [00:00<00:00, 175.48it/s]
2026-06-13 18:03:38.372 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 0.5622 test 0.4630 metric ['0.8359']
100%|██████████| 100/100 [00:00<00:00, 176.16it/s]
2026-06-13 18:03:39.185 | INFO     | mltrainer.trainer:report:209 - Epoch 3 train 0.5194 test 0.5034 metric ['0.8261']
2026-06-13 18:03:39.186 | INFO     | mltrainer.trainer:__call__:252 - best loss: 0.4630, current loss 0.5034.Counter 1/10.
100%|██████████| 100/100 [00:00<00:00, 176.19it/s]
2026-06-13 18:03:39.997 | INFO     | mltrainer.trainer:report:209 - Epoch 4 train 0.4819 test 0.4823 metric ['0.8242']
2026-06-13 18:03:39.998 | I

2026-06-13 18:04:00.308389 Finished run: rerun_top3_256_128_64_dr0.2_normbatch_seed3
All reruns completed
